## Translate contexts

1) Read lines from SemCor file
2) Filter by lemmas' counts(every lemma was included in train_set less than 8 times) 
3) Exclude ambiguous cases and words from validation set 
4) Translate via Deepseek
5) Save in JSON file

In [1]:
import pandas as pd

df = pd.read_csv('../gold_data/eng_gold.csv')
options = df.word

In [2]:
from nltk import WordNetLemmatizer as wnl

# set of nouns from validation set
test_nouns = set()
for option in options:
    lemma = wnl().lemmatize(option, 'n')
    test_nouns.add(lemma)

In [3]:
import json

def read_jsonl(file_path, cnt=35_000):
    """
        Read from jsonl file
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        read_lines = 0
        for line in f:
            line = line.strip()
            if line:
                read_lines += 1
                yield json.loads(line)
            if cnt == read_lines:
                break

# Использование:
lines_reader = read_jsonl('../wsd-data/data/jsonl/SemCor.jsonl', cnt=10e6)

In [4]:
lines = []
# list of lines from SemCor
for line in lines_reader:
    lines.append(line)

In [ ]:
from collections import Counter

cnt = Counter()
train_set = []

for line in lines:
    # filter lines
    if cnt[line['lemma']] < 8 and line['pos'] == 'NOUN' and line['lemma'] not in test_nouns and len(line['sense'].split(';')) == 1:
        train_set.append(line)
        cnt[line['lemma']] += 1

In [ ]:
def mask_target(sentence: str, start: int, end: int) -> str:
    """
        Insert placeholders to sentence
    """
    tokens = sentence.split()
    tokens.insert(end, "[TGT]")
    tokens.insert(start, "[TGT]")
    return " ".join(tokens)

from nltk.corpus import wordnet as wn

def get_gloss(sense_key: str) -> str:
    """
        Get gloss from WordNet sense key
    """
    lemma = wn.lemma_from_key(sense_key)
    return lemma.synset().definition()

# gloss_en = "the products of human creativity; works of art collectively"

In [ ]:
def prepare_for_translation(records):
    """
        Apply functuions mask_target and get_gloss for each sentence in filtered lines
    """
    contexts_en, glosses_en = [], []
    for rec in records:
        masked = mask_target(rec["sentence"], rec["start"], rec["end"])
        gloss = get_gloss(rec["sense"])
        contexts_en.append(masked)
        glosses_en.append(gloss)
    return contexts_en, glosses_en

contexts, glosses = prepare_for_translation(train_set)

In [107]:
contexts[10]

'Do you measure its relation to reduced absenteeism , [TGT] turnover [TGT] , accidents , and grievances , and to improved quality and output ?'

In [109]:
glosses_unique = list(set(glosses))

In [110]:
from openai import OpenAI
from dotenv import load_dotenv
import os
load_dotenv()
key = os.getenv("DEEPSEEK")

client = OpenAI(
    api_key=key,
    base_url="https://api.deepseek.com",
)

In [ ]:
import openai

def translate_batch(texts: list[str], source="English", target="Kyrgyz") -> list[str]:
    """
        Make API call to Deepseek to translate batch of sentences
    """
    prompt = (
        f"Translate each of the following English lines to Kyrgyz."
        "The tokens [TGT] are special markers that surround the target word." 
        "They MUST remain in the translation exactly as they are, and they MUST surround the translated target word."
        "DO NOT translate the [TGT] markers themselves."
        "Return the translations separated by ' ||| ' in the same order, preserving all punctuation and the exact placement of [TGT] around the appropriate word."
    )
    batch_text = " ||| ".join(texts)
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": batch_text}
    ]

    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=messages,
        temperature=0.0
    )

    translated = resp.choices[0].message.content.split(" ||| ")

    if len(translated) != len(texts):
        raise ValueError("Количество переводов не совпало с исходным")
    
    translated_batch = [t.strip() for t in translated]
    
    return translated_batch

In [ ]:
from tqdm import trange

translated_contexts = []
problem_index = []
batch_size = 20

for i in trange(0, len(contexts), batch_size):
    try:
        res = translate_batch(contexts[i:i+batch_size])
        translated_contexts.extend(res)
    except ValueError as e:
        # if error happens add empty lines to save structure corresponding contects list
        translated_contexts.extend([""]*len(contexts[i:i+batch_size]))
        problem_index.append(i)


In [116]:
len(problem_index) * batch_size

8820

In [ ]:
import tqdm

# review all sentences and translate problem sentences alone without batching
for i in tqdm.trange(len(translated_contexts)):
    cur_sent = translated_contexts[i]
    splt = cur_sent.split("[TGT]")
    # sentence should be splitted in 3 parts when 2 placeholders are correctly transferred between sentences
    # also sentence maybe empty if there was problem with its batch
    if len(splt) != 3 or cur_sent == "":
        prompt = (
            f"Translate the following English line to Kyrgyz."
            "The tokens [TGT] are special markers that surround the target word." 
            "They MUST remain in the translation exactly as they are, and they MUST surround the translated target word."
            "DO NOT translate the [TGT] markers themselves."
            "Return the translationpreserving all punctuation and the exact placement of [TGT] around the appropriate word."
        )
        messages = [
            {"role": "system", "content": prompt},
            {"role": "user", "content": contexts[i]}
        ]

        # repeat call
        resp = client.chat.completions.create(
            model="deepseek-chat",
            messages=messages,
            temperature=0.0
        )

        translated = resp.choices[0].message.content
    
        translated_contexts[i] = translated.strip()

In [148]:
len(translated_contexts)

34903

In [165]:
empty = 0
bad_split = 0
for l in translated_contexts:
    if l == "":
        empty += 1
    elif len(l.split("[TGT]")) != 3:
        bad_split += 1

print(empty)
print(bad_split)

0
52


In [ ]:
with open('kyr_sentences.json', 'w', encoding="utf-8") as f:
    json.dump(translated_contexts, f, ensure_ascii=False, indent=4)